In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np
from typing import List
import torch

device = "cpu"

model_name = "microsoft/deberta-large-mnli"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)


Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [8]:
def get_entailment_matrix(responses: List[str], debug=False):
    n = len(responses)
    similarity_matrix = np.zeros((n, n))
    
    entailment_id = -1
    for k, v in model.config.id2label.items():
        if v.upper() == "ENTAILMENT":
            entailment_id = k
            break
            
    if entailment_id == -1:
        raise ValueError("Could not find ENTAILMENT label in model config.")

    # TOKENIZATION LOOP

    tokenized_inputs_list = []
    
    print(f"Tokenizing {n*n} pairs individually...")
    
    for i in range(n):
        for j in range(n):

            current_pair = [responses[i], responses[j]]

            inputs = tokenizer(
                text=responses[i], 
                text_pair=responses[j],
                return_tensors="pt", 
                truncation=False, 
                max_length=256
            )

            if i == 0 and j ==4 and debug:
                print("\n" + "!"*40)
                print(f" DEBUG: TOKENIZATION CHECK (Pair {i} vs {j})")
                print("!"*40)
                print(f"Original 1: {current_pair[0]}")
                print(f"Original 2: {current_pair[1]}")
                

                decoded_sequence = tokenizer.decode(inputs["input_ids"][0])
                print(f"\nModel Input View:\n{decoded_sequence}")
                print("!"*40 + "\n")
            
            if i == 4 and j == 0 and debug:
                print("\n" + "!"*40)
                print(f" DEBUG: TOKENIZATION CHECK (Pair {i} vs {j})")
                print("!"*40)
                print(f"Original 1: {current_pair[0]}")
                print(f"Original 2: {current_pair[1]}")
                

                decoded_sequence = tokenizer.decode(inputs["input_ids"][0])
                print(f"\nModel Input View:\n{decoded_sequence}")
                print("!"*40 + "\n")

            tokenized_inputs_list.append((i, j, inputs))


    print("Running inference on tokenized list...")
    
    model.eval()
    
    for row, col, inputs in tokenized_inputs_list:

        inputs = inputs.to(device)

        with torch.no_grad():
            outputs = model(**inputs)
        

        probs = torch.nn.functional.softmax(outputs.logits, dim=1)
        

        entailment_score = probs[0][entailment_id].item()
        

        similarity_matrix[row, col] = entailment_score

    return similarity_matrix

In [6]:
my_responses = [
        "Global warming is a hoax to hurt the economy.",# Immediate action is required for the climate.", # Cluster 0
        "We can wait 50 years to see if weather stabilizes.", # Cluster 3
        "The stock market is fluctuating due to tech stocks.", # Cluster 1
        "Climate change demands urgent policy changes.", # Cluster 0
        "Governments must act now to stop global warming.", # Cluster 2
        "Immediate action is required for the climate.", # Cluster 0
    ]


# entailment_matrix = get_entailment_matrix(my_responses)
# for row in entailment_matrix:
#     print([f"{x:.3f}" for x in row])

In [11]:
statement2 = "I own a car."
statement1 = "I own a Honda civic."

# tokenize the statements
inputs = tokenizer(
    text=statement1,
    text_pair=statement2,
    return_tensors="pt",
    truncation=False,
    max_length=256
)

# detokenize the inputs
decoded_sequence = tokenizer.decode(inputs["input_ids"][0])
print(decoded_sequence)

# recompute the entailment score
with torch.no_grad():
    outputs = model(**inputs)

entailment_id = -1

probs = torch.nn.functional.softmax(outputs.logits, dim=1)
entailment_score = probs[0][entailment_id].item()
print(probs)
print(f"Entailment score: {entailment_score:.3f}")



[CLS]I own a Honda civic.[SEP]I own a car.[SEP]
tensor([[2.6769e-04, 4.2530e-03, 9.9548e-01]])
Entailment score: 0.995
